# LeetCode 1021 — Remove Outermost Parentheses bằng Python

**Link đề:** https://leetcode.com/problems/remove-outermost-parentheses/
**Độ khó:** Easy · **Chủ đề:** String, Stack, Bracket Sequences (dãy ngoặc)

```python
class Solution:
    def removeOuterParentheses(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu định nghĩa **chuỗi ngoặc hợp lệ** và **chuỗi nguyên thủy (primitive)**.
2. Biết đo **độ sâu (depth / balance)** của ngoặc bằng một biến đếm.
3. Ôn công cụ: duyệt chuỗi, slicing, `list` dùng như **stack**, `"".join`, vì sao không nên `+=` chuỗi trong vòng lặp dài.
4. Viết lời giải từ dễ đến gọn: tách nguyên thủy rồi cắt → stack → một biến đếm độ sâu.
5. Trace từng ký tự, chứng minh, phân tích độ phức tạp, so sánh các cách.
6. Làm bài tập biến thể: độ sâu lớn nhất, tách nguyên thủy, kiểm tra hợp lệ.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

**Chuỗi ngoặc hợp lệ** là: chuỗi rỗng `""`, hoặc `"(" + A + ")"`, hoặc `A + B`, với `A`, `B` là chuỗi ngoặc hợp lệ.
Ví dụ: `""`, `"()"`, `"(())()"`, `"(()(()))"`.

Một chuỗi hợp lệ `s` là **nguyên thủy (primitive)** nếu nó khác rỗng và **không** thể tách thành `A + B` với `A`, `B` đều khác rỗng và hợp lệ. Ví dụ `"(()())"` là nguyên thủy; `"()()"` thì không (`"()" + "()"`).

Mọi chuỗi hợp lệ tách được duy nhất thành `P1 + P2 + ... + Pk` (các nguyên thủy). **Yêu cầu:** bỏ cặp ngoặc **ngoài cùng** của **mỗi** `Pi` rồi ghép lại.

| `s` | Phân tách nguyên thủy | Bỏ ngoặc ngoài | Output |
|---|---|---|---|
| `"(()())(())"` | `"(()())" + "(())"` | `"()()" + "()"` | `"()()()"` |
| `"(()())(())(()(()))"` | `"(()())" + "(())" + "(()(()))"` | `"()()" + "()" + "()(())"` | `"()()()()(())"` |
| `"()()"` | `"()" + "()"` | `"" + ""` | `""` |

**Constraints:** `1 <= s.length <= 10^5`, `s[i]` là `'('` hoặc `')'`, `s` là chuỗi ngoặc hợp lệ.

In [1]:
s = "(()())(())"
P1, P2 = s[0:6], s[6:10]
print(P1, P2)
print("bỏ ngoài:", P1[1:-1], P2[1:-1], "->", P1[1:-1] + P2[1:-1])

(()()) (())
bỏ ngoài: ()() () -> ()()()


`P[1:-1]` là slicing bỏ ký tự **đầu** (chỉ số 0) và ký tự **cuối** (chỉ số `-1`). Đó chính là "bỏ cặp ngoặc ngoài cùng" của một nguyên thủy.

Câu hỏi tự đặt:
- Làm sao biết một nguyên thủy **kết thúc** ở đâu?
- Một dấu `(` là "ngoài cùng" khi nào? Một dấu `)` là "ngoài cùng" khi nào?

## 2. Nền tảng: độ sâu (balance) của ngoặc

Đi từ trái sang phải, giữ `depth`:
- gặp `(` → `depth += 1`
- gặp `)` → `depth -= 1`

Với chuỗi hợp lệ: `depth` không bao giờ âm và kết thúc bằng 0. **Mỗi lần `depth` trở về 0 là một nguyên thủy vừa kết thúc.**

In [2]:
s = "(()())(())"
depth = 0
for i, ch in enumerate(s):
    depth += 1 if ch == "(" else -1
    note = "<- depth về 0: kết thúc một nguyên thủy" if depth == 0 else ""
    print(f"i={i:>2} ch={ch} depth={depth} {note}")

i= 0 ch=( depth=1 
i= 1 ch=( depth=2 
i= 2 ch=) depth=1 
i= 3 ch=( depth=2 
i= 4 ch=) depth=1 
i= 5 ch=) depth=0 <- depth về 0: kết thúc một nguyên thủy
i= 6 ch=( depth=1 
i= 7 ch=( depth=2 
i= 8 ch=) depth=1 
i= 9 ch=) depth=0 <- depth về 0: kết thúc một nguyên thủy


`depth += 1 if ch == "(" else -1` dùng **biểu thức điều kiện**: cộng `1` nếu là `(`, ngược lại cộng `-1`.

### 2.1. Hình dung bằng "núi"

Vẽ `depth` theo vị trí thành một dãy núi: mỗi nguyên thủy là một ngọn núi từ mặt đất (0) lên rồi về mặt đất. Ngoặc ngoài cùng là **bước lên đầu tiên từ mặt đất** và **bước xuống cuối cùng về mặt đất**.

In [3]:
def draw_mountain(s):
    depth = 0
    heights = []
    for ch in s:
        if ch == "(":
            depth += 1
            heights.append(depth)        # chiều cao SAU khi lên
        else:
            heights.append(depth)        # chiều cao TRƯỚC khi xuống
            depth -= 1
    top = max(heights)
    for level in range(top, 0, -1):
        print("".join(ch if h == level else " " for ch, h in zip(s, heights)))
    print("-" * len(s))

draw_mountain("(()())(())(()(()))")

              ()  
 ()()  ()  ()(  ) 
(    )(  )(      )
------------------


- `zip(s, heights)` ghép từng ký tự với chiều cao của nó thành cặp.
- `range(top, 0, -1)` đếm ngược từ `top` xuống 1.
- Dòng thấp nhất (level 1) chứa đúng các **ngoặc ngoài cùng** — đó là những ký tự ta cần bỏ!

### 2.2. `list` làm stack

Stack (ngăn xếp) là cấu trúc "vào sau ra trước". Trong Python dùng `list`: `append(x)` để đẩy vào, `pop()` để lấy ra phần tử cuối, `stack[-1]` để xem đỉnh, `len(stack)` là kích thước.

In [4]:
stack = []
stack.append("(")
stack.append("(")
print("sau 2 lần push:", stack, "| size =", len(stack))
top = stack.pop()
print("pop ra:", top, "| còn:", stack)
print("stack rỗng?", not stack, "| sau pop nữa:", (stack.pop(), stack), "| rỗng?", not stack)

sau 2 lần push: ['(', '('] | size = 2
pop ra: ( | còn: ['(']
stack rỗng? False | sau pop nữa: ('(', []) | rỗng? True


`not stack` là `True` khi list rỗng (list rỗng được coi là "sai" trong ngữ cảnh điều kiện).

Với chuỗi ngoặc, **kích thước stack chính là `depth`**. Vì vậy ta có thể dùng biến đếm thay cho stack.

### 2.3. Ghép chuỗi: `+=` và `"".join(list)`

Chuỗi Python là **bất biến (immutable)**: `res += ch` có thể phải tạo chuỗi mới mỗi lần. Cách chuẩn là gom ký tự vào `list` rồi `"".join(list)` một lần ở cuối — O(n) đảm bảo.

In [5]:
parts = []
for ch in "abc":
    parts.append(ch)
print(parts, "->", "".join(parts))

['a', 'b', 'c'] -> abc


(CPython có tối ưu giúp `+=` nhanh trong nhiều trường hợp, nhưng không được đảm bảo; quen dùng `join` sẽ an toàn ở mọi nơi.)

## 3. Cách trực giác / Baseline

### 3.1. Cách 1 — Tách nguyên thủy bằng cách đếm `(` và `)` trên từng tiền tố (O(n²))

Ý tưởng ngây thơ: từ vị trí `start`, mở rộng `end` dần cho tới khi `s[start:end]` có số `(` bằng số `)` → đó là một nguyên thủy. Dùng `str.count` mỗi lần → chậm.

In [6]:
def remove_outer_naive(s: str) -> str:
    result = []
    start = 0
    n = len(s)
    while start < n:
        end = start + 1
        while s[start:end].count("(") != s[start:end].count(")"):
            end += 1
        primitive = s[start:end]
        result.append(primitive[1:-1])
        start = end
    return "".join(result)

print(repr(remove_outer_naive("(()())(())")))
print(repr(remove_outer_naive("(()())(())(()(()))")))
print(repr(remove_outer_naive("()()")))

'()()()'
'()()()()(())'
''


`repr(x)` in chuỗi kèm dấu nháy — giúp **nhìn thấy chuỗi rỗng** `''` (nếu chỉ `print("")` thì ra một dòng trống, khó nhận ra).

Mỗi lần `count` trên đoạn dài `L` tốn O(L); với nguyên thủy dài `L`, vòng trong tốn `1 + 2 + ... + L = O(L²)` → tệ nhất O(n²) khi cả chuỗi là một nguyên thủy lớn.

### 3.2. Cách 2 — Tách nguyên thủy bằng `depth`, rồi cắt `[1:-1]` (O(n))

Thay vì đếm lại, ta dùng `depth` chạy (mục 2): khi `depth` về 0 tại `i`, nguyên thủy là `s[start:i+1]`.

In [7]:
def remove_outer_split(s: str) -> str:
    result = []
    depth = 0
    start = 0
    for i, ch in enumerate(s):
        depth += 1 if ch == "(" else -1
        if depth == 0:
            result.append(s[start + 1:i])     # bỏ s[start] và s[i]
            start = i + 1
    return "".join(result)

print(repr(remove_outer_split("(()())(())")))
print(repr(remove_outer_split("(()())(())(()(()))")))
print(repr(remove_outer_split("()()")))

'()()()'
'()()()()(())'
''


`s[start + 1:i]` lấy từ sau ngoặc mở ngoài cùng (`start`) tới **trước** ngoặc đóng ngoài cùng (`i`) — đúng phần ruột của nguyên thủy.

## 4. Nút thắt & insight

Cách 2 đã O(n), nhưng vẫn phải nhớ `start` và cắt chuỗi. Có cách nào quyết định **từng ký tự một** là giữ hay bỏ?

Nhìn hình núi ở mục 2.1:
- `(` là ngoài cùng ⟺ **trước khi** gặp nó `depth == 0` (đang ở mặt đất).
- `)` là ngoài cùng ⟺ **sau khi** gặp nó `depth == 0` (về lại mặt đất).

Vậy quy tắc giữ ký tự:
- Với `(`: giữ nếu `depth > 0` **trước** khi tăng.
- Với `)`: giữ nếu `depth > 0` **sau** khi giảm.

Thứ tự "kiểm tra rồi tăng" / "giảm rồi kiểm tra" là **mấu chốt** của bài.

In [8]:
s = "(()())"
depth = 0
for ch in s:
    if ch == "(":
        print(f"'(' depth trước = {depth} -> {'GIỮ' if depth > 0 else 'BỎ (ngoài cùng)'}")
        depth += 1
    else:
        depth -= 1
        print(f"')' depth sau  = {depth} -> {'GIỮ' if depth > 0 else 'BỎ (ngoài cùng)'}")

'(' depth trước = 0 -> BỎ (ngoài cùng)
'(' depth trước = 1 -> GIỮ
')' depth sau  = 1 -> GIỮ
'(' depth trước = 1 -> GIỮ
')' depth sau  = 1 -> GIỮ
')' depth sau  = 0 -> BỎ (ngoài cùng)


## 5. Lời giải chính — một biến đếm độ sâu

```python
res = []                     # Đoạn 1: nơi gom ký tự được giữ
depth = 0
for ch in s:                 # Đoạn 2: duyệt từng ký tự
    if ch == "(":
        if depth > 0:        #   không phải ngoặc mở ngoài cùng
            res.append(ch)
        depth += 1           #   tăng SAU khi kiểm tra
    else:
        depth -= 1           #   giảm TRƯỚC khi kiểm tra
        if depth > 0:        #   không phải ngoặc đóng ngoài cùng
            res.append(ch)
return "".join(res)          # Đoạn 3: ghép
```

In [9]:
def remove_outer_depth(s: str) -> str:
    res = []
    depth = 0
    for ch in s:
        if ch == "(":
            if depth > 0:
                res.append(ch)
            depth += 1
        else:
            depth -= 1
            if depth > 0:
                res.append(ch)
    return "".join(res)

for s in ["(()())(())", "(()())(())(()(()))", "()()"]:
    print(f"{s!r:>22} -> {remove_outer_depth(s)!r}")

          '(()())(())' -> '()()()'
  '(()())(())(()(()))' -> '()()()()(())'
                '()()' -> ''


`{s!r:>22}` trong f-string: `!r` dùng `repr` (có dấu nháy), `:>22` căn phải độ rộng 22.

### 5.1. Cách 3 — Dùng stack thật (cho người quen bài "Valid Parentheses")

Đẩy `(` vào stack, gặp `)` thì pop. Ký tự được giữ nếu stack **khác rỗng** tại thời điểm tương ứng. Logic y hệt cách đếm — chỉ là `len(stack)` thay cho `depth`.

In [10]:
def remove_outer_stack(s: str) -> str:
    res = []
    stack = []
    for ch in s:
        if ch == "(":
            if stack:                 # đã có ngoặc mở bao ngoài
                res.append(ch)
            stack.append(ch)
        else:
            stack.pop()
            if stack:                 # vẫn còn ngoặc bao ngoài
                res.append(ch)
    return "".join(res)

for s in ["(()())(())", "(()())(())(()(()))", "()()"]:
    print(f"{s!r:>22} -> {remove_outer_stack(s)!r}")

          '(()())(())' -> '()()()'
  '(()())(())(()(()))' -> '()()()()(())'
                '()()' -> ''


### 5.2. Cách 4 — Viết gọn: giữ ký tự khi `depth` "phía trong" ≥ 1

Gộp hai nhánh: với `(` dùng độ sâu **trước** khi tăng; với `)` dùng độ sâu **sau** khi giảm. Cả hai đều chính là "độ sâu của cặp ngoặc chứa ký tự này, tính từ 0". Ngoặc ngoài cùng có độ sâu cặp = 0.

In [11]:
def remove_outer_compact(s: str) -> str:
    res = []
    depth = 0
    for ch in s:
        if ch == ")":
            depth -= 1
        if depth:                 # depth != 0 -> không phải ngoài cùng
            res.append(ch)
        if ch == "(":
            depth += 1
    return "".join(res)

for s in ["(()())(())", "(()())(())(()(()))", "()()"]:
    print(f"{s!r:>22} -> {remove_outer_compact(s)!r}")

          '(()())(())' -> '()()()'
  '(()())(())(()(()))' -> '()()()()(())'
                '()()' -> ''


Thứ tự ba câu lệnh: **giảm (nếu `)`) → kiểm tra → tăng (nếu `(`)**. Đây là cách viết ngắn nhất, nhưng hãy hiểu Cách chính trước rồi mới rút gọn.

## 6. TRACE từng bước

Ví dụ `s = "(()())(())"`.

| i | ch | depth trước | thao tác | depth sau | giữ? | res |
|:-:|:-:|:-:|---|:-:|:-:|---|
| 0 | ( | 0 | kiểm tra 0 > 0 ✗, tăng | 1 | ✗ | `` |
| 1 | ( | 1 | 1 > 0 ✓, tăng | 2 | ✓ | `(` |
| 2 | ) | 2 | giảm, 1 > 0 ✓ | 1 | ✓ | `()` |
| 3 | ( | 1 | 1 > 0 ✓, tăng | 2 | ✓ | `()(` |
| 4 | ) | 2 | giảm, 1 > 0 ✓ | 1 | ✓ | `()()` |
| 5 | ) | 1 | giảm, 0 > 0 ✗ | 0 | ✗ | `()()` |
| 6 | ( | 0 | 0 > 0 ✗, tăng | 1 | ✗ | `()()` |
| 7 | ( | 1 | ✓, tăng | 2 | ✓ | `()()(` |
| 8 | ) | 2 | giảm, ✓ | 1 | ✓ | `()()()` |
| 9 | ) | 1 | giảm, ✗ | 0 | ✗ | `()()()` |

In [12]:
def trace_remove_outer(s):
    res = []
    depth = 0
    print(f"{'i':>2} | ch | depth trước | depth sau | giữ? | res")
    for i, ch in enumerate(s):
        before = depth
        if ch == "(":
            keep = depth > 0
            depth += 1
        else:
            depth -= 1
            keep = depth > 0
        if keep:
            res.append(ch)
        print(f"{i:>2} | {ch}  |      {before}      |     {depth}     | {'✓' if keep else '✗'}    | {''.join(res)}")
    return "".join(res)

print("Kết quả:", repr(trace_remove_outer("(()())(())")))

 i | ch | depth trước | depth sau | giữ? | res
 0 | (  |      0      |     1     | ✗    | 
 1 | (  |      1      |     2     | ✓    | (
 2 | )  |      2      |     1     | ✓    | ()
 3 | (  |      1      |     2     | ✓    | ()(
 4 | )  |      2      |     1     | ✓    | ()()
 5 | )  |      1      |     0     | ✗    | ()()
 6 | (  |      0      |     1     | ✗    | ()()
 7 | (  |      1      |     2     | ✓    | ()()(
 8 | )  |      2      |     1     | ✓    | ()()()
 9 | )  |      1      |     0     | ✗    | ()()()
Kết quả: '()()()'


### 6.1. Trace ca toàn nguyên thủy nhỏ `"()()"` — kết quả rỗng

In [13]:
print("Kết quả:", repr(trace_remove_outer("()()")))

 i | ch | depth trước | depth sau | giữ? | res
 0 | (  |      0      |     1     | ✗    | 
 1 | )  |      1      |     0     | ✗    | 
 2 | (  |      0      |     1     | ✗    | 
 3 | )  |      1      |     0     | ✗    | 
Kết quả: ''


## 7. Code nộp LeetCode (Python 3)

In [14]:
class Solution:
    def removeOuterParentheses(self, s: str) -> str:
        res = []
        depth = 0
        for ch in s:
            if ch == "(":
                if depth > 0:
                    res.append(ch)
                depth += 1
            else:
                depth -= 1
                if depth > 0:
                    res.append(ch)
        return "".join(res)

sol = Solution()
assert sol.removeOuterParentheses("(()())(())") == "()()()"
assert sol.removeOuterParentheses("(()())(())(()(()))") == "()()()()(())"
assert sol.removeOuterParentheses("()()") == ""
print("Solution PASS 3 ví dụ của đề")

Solution PASS 3 ví dụ của đề


Phiên bản tách nguyên thủy (Cách 2) để nộp:

In [15]:
class SolutionSplit:
    def removeOuterParentheses(self, s: str) -> str:
        result = []
        depth = 0
        start = 0
        for i, ch in enumerate(s):
            depth += 1 if ch == "(" else -1
            if depth == 0:
                result.append(s[start + 1:i])
                start = i + 1
        return "".join(result)

assert SolutionSplit().removeOuterParentheses("(()())(())(()(()))") == "()()()()(())"
print("SolutionSplit PASS")

SolutionSplit PASS


## 8. Vì sao đúng?

**Bất biến:** trước khi xử lý ký tự `s[i]`, `depth` = (số `(`) − (số `)`) trong `s[0..i-1]` = số ngoặc mở **chưa được đóng**.

- Vì `s` hợp lệ, `depth ≥ 0` luôn đúng và các thời điểm `depth = 0` chính là **ranh giới** giữa các nguyên thủy (một nguyên thủy kết thúc ngay khi số mở bằng số đóng lần đầu, nếu không nó sẽ tách được thành `A + B`).
- Một `(` ở vị trí `i` là ngoặc mở ngoài cùng của một nguyên thủy ⟺ nó bắt đầu nguyên thủy mới ⟺ `depth = 0` trước nó.
- Một `)` ở vị trí `i` là ngoặc đóng ngoài cùng ⟺ nó kết thúc nguyên thủy ⟺ `depth = 0` sau nó.
- Mọi ký tự khác đều nằm **bên trong** một nguyên thủy và được giữ đúng thứ tự.

Do đó output = nối các `Pi[1:-1]` theo thứ tự — đúng yêu cầu.

**Hệ quả kiểm chứng được:** độ dài output = `len(s) - 2·k`, với `k` là số nguyên thủy (= số lần `depth` về 0).

In [16]:
def count_primitives(s):
    depth = cnt = 0
    for ch in s:
        depth += 1 if ch == "(" else -1
        cnt += depth == 0
    return cnt

for s in ["(()())(())", "(()())(())(()(()))", "()()"]:
    k = count_primitives(s)
    out = Solution().removeOuterParentheses(s)
    print(f"{s!r}: k={k}, len(s)-2k={len(s) - 2 * k}, len(out)={len(out)}")
    assert len(out) == len(s) - 2 * k

'(()())(())': k=2, len(s)-2k=6, len(out)=6
'(()())(())(()(()))': k=3, len(s)-2k=12, len(out)=12
'()()': k=2, len(s)-2k=0, len(out)=0


## 9. Phân tích Time / Space

Gọi `n = len(s)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm |
|---|---|---|---|
| 1. Đếm lại từng tiền tố | `count` mỗi lần mở rộng | `O(n²)` | `O(n)` |
| 2. Tách bằng `depth` + slicing | một lượt + cắt | `O(n)` | `O(n)` (output) |
| 3. Stack | `len(stack)` = depth | `O(n)` | `O(n)` (stack + output) |
| Chính / 4. Biến đếm | quyết định từng ký tự | `O(n)` | `O(1)` ngoài output |

Output có thể dài tới `n - 2` → bộ nhớ cho kết quả luôn O(n); "bộ nhớ thêm" của cách biến đếm chỉ là O(1).

## 10. Kiểm thử: mẫu đề, biên, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS ({actual!r})")

cases = [
    ("đề 1", "(()())(())", "()()()"),
    ("đề 2", "(()())(())(()(()))", "()()()()(())"),
    ("đề 3", "()()", ""),
    ("ngắn nhất", "()", ""),
    ("một nguyên thủy lồng sâu", "((()))", "(())"),
    ("lồng sâu + kề nhau", "((()))(())", "(())()"),
    ("nguyên thủy chứa 2 khối", "(()())", "()()"),
    ("nhiều () liên tiếp", "()" * 5, ""),
]
for name, s, exp in cases:
    check(name, Solution().removeOuterParentheses(s), exp)

đề 1                               PASS ('()()()')
đề 2                               PASS ('()()()()(())')
đề 3                               PASS ('')
ngắn nhất                          PASS ('')
một nguyên thủy lồng sâu           PASS ('(())')
lồng sâu + kề nhau                 PASS ('(())()')
nguyên thủy chứa 2 khối            PASS ('()()')
nhiều () liên tiếp                 PASS ('')


### 10.1. Sinh chuỗi ngoặc hợp lệ ngẫu nhiên và đối chiếu mọi cách

Để sinh chuỗi hợp lệ ngẫu nhiên: ở mỗi bước, được thêm `(` nếu còn quota mở, được thêm `)` nếu `depth > 0`.

In [18]:
import random
random.seed(1021)

def random_valid_parentheses(pairs: int) -> str:
    res = []
    opened = depth = 0
    while len(res) < 2 * pairs:
        can_open = opened < pairs
        can_close = depth > 0
        if can_open and (not can_close or random.random() < 0.5):
            res.append("(")
            opened += 1
            depth += 1
        else:
            res.append(")")
            depth -= 1
    return "".join(res)

solvers = {
    "naive": remove_outer_naive,
    "split": remove_outer_split,
    "depth": remove_outer_depth,
    "stack": remove_outer_stack,
    "compact": remove_outer_compact,
    "Solution": Solution().removeOuterParentheses,
    "SolutionSplit": SolutionSplit().removeOuterParentheses,
}
print("ví dụ chuỗi sinh ra:", random_valid_parentheses(6))
for _ in range(1000):
    s = random_valid_parentheses(random.randint(1, 15))
    expected = remove_outer_naive(s)
    for name, f in solvers.items():
        assert f(s) == expected, (name, s)
print("1000 chuỗi ngẫu nhiên: mọi cách khớp nhau")

ví dụ chuỗi sinh ra: ()((((()))))
1000 chuỗi ngẫu nhiên: mọi cách khớp nhau


`random.random()` trả số thực trong `[0, 1)`; `< 0.5` nghĩa là xác suất 50%.

### 10.2. Input lớn `n = 10^5`

In [19]:
import time
big = random_valid_parentheses(50_000)
for name in ["split", "depth", "stack", "compact"]:
    start = time.perf_counter()
    out = solvers[name](big)
    print(f"{name:<8}: len(out) = {len(out)}, {time.perf_counter() - start:.4f}s")

split   : len(out) = 99720, 0.0056s
depth   : len(out) = 99720, 0.0061s
stack   : len(out) = 99720, 0.0057s
compact : len(out) = 99720, 0.0073s


## 11. Sai lầm người mới hay mắc

### 11.1. Sai thứ tự: tăng `depth` trước khi kiểm tra `(`

In [20]:
def wrong_order_open(s):
    res = []
    depth = 0
    for ch in s:
        if ch == "(":
            depth += 1              # SAI: tăng trước
            if depth > 0:
                res.append(ch)
        else:
            depth -= 1
            if depth > 0:
                res.append(ch)
    return "".join(res)

print("sai :", repr(wrong_order_open("(()())(())")))
print("đúng:", repr(Solution().removeOuterParentheses("(()())(())")))

sai : '(()()(()'
đúng: '()()()'


**Vì sao sai:** sau khi tăng, `depth ≥ 1` luôn đúng → mọi `(` đều được giữ, kể cả ngoặc mở ngoài cùng. Output mất cân bằng. **Sửa:** kiểm tra `depth > 0` **trước** khi `depth += 1`.

### 11.2. Sai thứ tự: kiểm tra `)` trước khi giảm

In [21]:
def wrong_order_close(s):
    res = []
    depth = 0
    for ch in s:
        if ch == "(":
            if depth > 0:
                res.append(ch)
            depth += 1
        else:
            if depth > 0:           # SAI: kiểm tra trước khi giảm (depth luôn >= 1 ở đây)
                res.append(ch)
            depth -= 1
    return "".join(res)

print("sai :", repr(wrong_order_close("(()())(())")))
print("đúng:", repr(Solution().removeOuterParentheses("(()())(())")))

sai : '()())())'
đúng: '()()()'


**Vì sao sai:** khi gặp `)`, `depth` trước khi giảm luôn ≥ 1 → mọi `)` đều được giữ. **Sửa:** `depth -= 1` rồi mới kiểm tra.

### 11.3. Chỉ bỏ ký tự đầu và cuối của **cả chuỗi**

In [22]:
def wrong_strip_whole(s):
    return s[1:-1]

print("sai :", repr(wrong_strip_whole("(()())(())")))
print("đúng:", repr(Solution().removeOuterParentheses("(()())(())")))

sai : '()())(()'
đúng: '()()()'


**Vì sao sai:** đề yêu cầu bỏ ngoặc ngoài cùng của **từng** nguyên thủy, không phải của cả chuỗi. Output sai còn không hợp lệ: `"()())(()"`. **Sửa:** tách nguyên thủy trước.

### 11.4. Dùng `str.replace` hoặc xóa "cặp ngoài" theo mẫu cố định

Một số bạn thử `s.replace("()", "")` hoặc các mẹo chuỗi. Những mẹo này không phản ánh cấu trúc lồng nhau, ví dụ `"((()))"` cần ra `"(())"`, nhưng `replace("()", "")` ra `"(())"` một cách "may mắn"; với `"(()())"` cần ra `"()()"` thì `replace` cho `"()"`. Luôn dựa vào **độ sâu**.

In [23]:
for s in ["((()))", "(()())"]:
    print(f"{s}: replace -> {s.replace('()', '')!r} | đúng -> {Solution().removeOuterParentheses(s)!r}")

((())): replace -> '(())' | đúng -> '(())'
(()()): replace -> '()' | đúng -> '()()'


## 12. Nhận diện pattern

Dấu hiệu:
- Chuỗi chỉ gồm ngoặc, hỏi về "lồng nhau", "ngoài cùng", "độ sâu", "cân bằng" → nghĩ tới **biến đếm độ sâu** (balance) hoặc **stack**.
- Chỉ có **một loại** ngoặc → biến đếm đủ thay stack. Nhiều loại ngoặc `()[]{}` → cần stack để nhớ loại.
- "Tách thành các khối cân bằng" → mỗi lần `depth` về 0 là một ranh giới.

Bài liên quan:
- LeetCode 20 (Valid Parentheses) — stack nhiều loại ngoặc.
- LeetCode 1614 (Maximum Nesting Depth of the Parentheses) — max của `depth`.
- LeetCode 1221 (Split a String in Balanced Strings) — đếm số lần balance về 0.
- LeetCode 921 (Minimum Add to Make Parentheses Valid), 856 (Score of Parentheses), 1249 (Minimum Remove to Make Valid Parentheses).

## 13. Biến thể

### 13.1. Biến thể A — trả về danh sách các nguyên thủy

In [24]:
def primitive_decomposition(s: str) -> list[str]:
    parts = []
    depth = start = 0
    for i, ch in enumerate(s):
        depth += 1 if ch == "(" else -1
        if depth == 0:
            parts.append(s[start:i + 1])
            start = i + 1
    return parts

print(primitive_decomposition("(()())(())(()(()))"))

['(()())', '(())', '(()(()))']


### 13.2. Biến thể B — bỏ **mọi** ngoặc ở độ sâu ≤ `d` (bóc `d` lớp)

Với `d = 1` chính là bài gốc. Ký tự có độ sâu cặp (tính từ 0 như mục 5.2) `< d` thì bỏ.

In [25]:
def peel_layers(s: str, d: int) -> str:
    res = []
    depth = 0
    for ch in s:
        if ch == ")":
            depth -= 1
        if depth >= d:
            res.append(ch)
        if ch == "(":
            depth += 1
    return "".join(res)

s = "(()(()))((()))"
for d in range(0, 4):
    print(f"bóc {d} lớp: {peel_layers(s, d)!r}")

bóc 0 lớp: '(()(()))((()))'
bóc 1 lớp: '()(())(())'
bóc 2 lớp: '()()'
bóc 3 lớp: ''


## 14. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Với `s = "(()(()))()"`, viết dãy `depth` sau mỗi ký tự, tìm các nguyên thủy, và output của bài.

### Bài 2 — Dự đoán output
Đoán `peel_layers("((()))", 2)` và `count_primitives("()(())()")`.

### Bài 3 — TODO: viết lại lời giải biến đếm

In [26]:
def exercise_remove_outer(s: str) -> str:
    # TODO 1: res = [], depth = 0
    # TODO 2: '(' -> kiểm tra depth > 0 rồi mới tăng
    # TODO 3: ')' -> giảm rồi mới kiểm tra depth > 0
    pass

# assert exercise_remove_outer("(()())(())") == "()()()"
# assert exercise_remove_outer("()()") == ""

### Bài 4 — TODO: độ sâu lồng nhau lớn nhất (LeetCode 1614 bản chỉ có ngoặc)

In [27]:
def exercise_max_depth(s: str) -> int:
    # TODO
    pass

# assert exercise_max_depth("(()(()))") == 3
# assert exercise_max_depth("()()") == 1

### Bài 5 — TODO biến thể: kiểm tra chuỗi chỉ gồm `(` `)` có hợp lệ không (không dùng stack)

In [28]:
def exercise_is_valid(s: str) -> bool:
    # TODO: depth không bao giờ âm và kết thúc bằng 0
    pass

# assert exercise_is_valid("(()())") is True
# assert exercise_is_valid("())(") is False
# assert exercise_is_valid("((") is False

### Bài 6 — Mini challenge: chỉ giữ lại các nguyên thủy có độ sâu lớn nhất ≥ `d`

Ví dụ `s = "()(())((()))"`, `d = 2` → giữ `"(())"` và `"((()))"` → `"(())((()))"`. Yêu cầu một lượt duyệt O(n).

In [29]:
def exercise_keep_deep_primitives(s: str, d: int) -> str:
    # TODO
    pass

# assert exercise_keep_deep_primitives("()(())((()))", 2) == "(())((()))"
# assert exercise_keep_deep_primitives("()(())((()))", 3) == "((()))"
# assert exercise_keep_deep_primitives("()()", 2) == ""

## 15. Gợi ý ba mức

**Bài 1** — (1) `depth` tăng với `(`, giảm với `)`; (2) đếm các lần về 0; (3) có 2 nguyên thủy.

**Bài 2** — (1) bóc 2 lớp từ `((()))`; (2) mỗi lần depth về 0 là một nguyên thủy; (3) `"()"` và `3`.

**Bài 3** — (1) mục 5; (2) thứ tự "kiểm tra rồi tăng" / "giảm rồi kiểm tra"; (3) cuối cùng `"".join(res)`.

**Bài 4** — (1) cập nhật max sau mỗi `(`; (2) `best = max(best, depth)`; (3) chỉ cần một biến `depth`.

**Bài 5** — (1) nếu `depth < 0` ở bất kỳ đâu → `False`; (2) cuối cùng `depth == 0`; (3) `return False` sớm khi âm.

**Bài 6** — (1) theo dõi `start` của nguyên thủy hiện tại và `peak` = max depth trong nó; (2) khi `depth` về 0, nếu `peak >= d` thì giữ `s[start:i+1]`; (3) reset `peak = 0`, `start = i + 1`.

## 16. Đáp án tham khảo

**Bài 1:** `depth`: 1,2,1,2,3,2,1,0,1,0. Nguyên thủy: `"(()(()))"`, `"()"`. Output: `"()(())" + "" = "()(())"`.

**Bài 2:** `peel_layers("((()))", 2) = "()"`, `count_primitives("()(())()") = 3`.

In [30]:
def answer_remove_outer(s):
    res = []
    depth = 0
    for ch in s:
        if ch == "(":
            if depth > 0:
                res.append(ch)
            depth += 1
        else:
            depth -= 1
            if depth > 0:
                res.append(ch)
    return "".join(res)


def answer_max_depth(s):
    depth = best = 0
    for ch in s:
        if ch == "(":
            depth += 1
            best = max(best, depth)
        else:
            depth -= 1
    return best


def answer_is_valid(s):
    depth = 0
    for ch in s:
        depth += 1 if ch == "(" else -1
        if depth < 0:
            return False
    return depth == 0


def answer_keep_deep_primitives(s, d):
    res = []
    depth = peak = start = 0
    for i, ch in enumerate(s):
        depth += 1 if ch == "(" else -1
        peak = max(peak, depth)
        if depth == 0:
            if peak >= d:
                res.append(s[start:i + 1])
            start = i + 1
            peak = 0
    return "".join(res)


assert answer_remove_outer("(()(()))()") == "()(())"
assert peel_layers("((()))", 2) == "()" and count_primitives("()(())()") == 3
assert answer_remove_outer("(()())(())") == "()()()" and answer_remove_outer("()()") == ""
assert answer_max_depth("(()(()))") == 3 and answer_max_depth("()()") == 1
assert answer_is_valid("(()())") is True and answer_is_valid("())(") is False and answer_is_valid("((") is False
assert answer_keep_deep_primitives("()(())((()))", 2) == "(())((()))"
assert answer_keep_deep_primitives("()(())((()))", 3) == "((()))"
assert answer_keep_deep_primitives("()()", 2) == ""

random.seed(7)
for _ in range(500):
    s = random_valid_parentheses(random.randint(1, 12))
    assert answer_is_valid(s)
    assert answer_remove_outer(s) == peel_layers(s, 1)
    assert answer_is_valid(answer_remove_outer(s))          # output luôn hợp lệ
    expected_deep = "".join(p for p in primitive_decomposition(s) if answer_max_depth(p) >= 2)
    assert answer_keep_deep_primitives(s, 2) == expected_deep
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


Một tính chất thú vị được kiểm ở trên: **output luôn là chuỗi ngoặc hợp lệ** — vì bỏ cặp ngoài của `"(" + A + ")"` cho ra `A` hợp lệ.

## 17. Tổng kết — template + checklist

**Template nhớ nhanh:**

```python
class Solution:
    def removeOuterParentheses(self, s: str) -> str:
        res = []
        depth = 0
        for ch in s:
            if ch == "(":
                if depth > 0:
                    res.append(ch)
                depth += 1
            else:
                depth -= 1
                if depth > 0:
                    res.append(ch)
        return "".join(res)
```

**Câu thần chú:** *"Mở: xét rồi tăng — Đóng: giảm rồi xét — depth 0 là mặt đất, ngoặc chạm đất thì bỏ."*

**Checklist trước khi submit:**
- [ ] Với `(`: kiểm tra `depth > 0` **trước** khi tăng?
- [ ] Với `)`: giảm **trước** rồi mới kiểm tra?
- [ ] Gom vào `list` và `"".join` ở cuối?
- [ ] Bỏ ngoặc ngoài của **từng** nguyên thủy, không phải cả chuỗi?
- [ ] Đã test `"()()"` (output rỗng) và `"((()))"` (lồng sâu)?